# pola: Critical Bandwidth for Bimodal Distributions

**pola** is a Python package for detecting whether a distribution is **meaningfully bimodal**
using the **critical bandwidth** method in kernel density estimation (KDE).

The critical bandwidth is the smallest bandwidth $h$ where the KDE transitions from
bimodal (2+ peaks) to unimodal (1 peak). Bandwidths below this threshold produce
a bimodal density estimate, while bandwidths above it smooth out the second mode.

This notebook walks through three visualizations that illustrate the core algorithm:
1. **Bandwidth Sweep** — KDE at 4 different bandwidths
2. **Critical Transition** — The bimodal-to-unimodal transition at $h_{\text{crit}}$
3. **Component Decomposition** — Splitting the distribution into two Gaussians

In [ ]:
%matplotlib inline

import sys

import numpy as np
from scipy import signal
from scipy import stats as scipy_stats

from pola import (
    critical_bandwidth,
    detect_components,
    find_trough,
    gaussian_kde,
    silverman_bandwidth,
)
from pola.bandwidth import count_modes
from pola.benchmark import BENCHMARK_CASES

try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib is required. Install with: uv sync --group viz")
    sys.exit(0)

print("All imports successful.")

## Data Generation

We use a standard benchmark case: two well-separated Gaussians with equal variance.

$$
X \sim 0.5 \cdot N(\mu=-2, \sigma=0.3) + 0.5 \cdot N(\mu=+2, \sigma=0.3)
$$

Total: 400 data points.

In [ ]:
case = BENCHMARK_CASES["well_separated_equal_var"]
x = case.generator(42)
h_silver = silverman_bandwidth(x)
h_crit, ok = critical_bandwidth(x)

print(f"Data: {len(x)} points from {case.name}")
print(f"Silverman bandwidth:  {h_silver:.4f}")
print(f"Critical bandwidth:   {h_crit:.4f}")
if not ok:
    print("  (critical bandwidth search did not fully converge)")

trough = find_trough(x, h_crit * 0.85, refine=True)
if trough is not None:
    print(f"Trough position:      {trough:.4f}")

result = detect_components(x)
print(
    f"Component 1: mean={result.component1.mean:+.3f}, std={result.component1.std:.3f}, weight={result.component1.weight:.3f}"
)
print(
    f"Component 2: mean={result.component2.mean:+.3f}, std={result.component2.std:.3f}, weight={result.component2.weight:.3f}"
)
print(f"Dip ratio: {result.dip_ratio:.4f}")

## Three Visualizations

### 1. KDE Bandwidth Sweep

A 2×2 grid showing KDE at four different bandwidths:
- **h = 0.05** — undersmoothed, many spurious peaks
- **h = Silverman** — reference bandwidth (rule-of-thumb)
- **h = h_crit** — critical bandwidth (just barely unimodal)
- **h = h_crit × 3** — oversmoothed, peak obscured

Red triangles mark peaks; green circles mark troughs.

In [ ]:
def _peak_trough_marks(x, h, grid, kde_vals, prominence=0.01):
    """Return (peak_x, trough_x) for the two highest KDE peaks."""
    peaks = signal.find_peaks(kde_vals, prominence=prominence * np.max(kde_vals))[0]
    if len(peaks) < 2:
        return None, None
    top_two = np.argsort(kde_vals[peaks])[-2:]
    pi = np.sort(peaks[top_two])
    min_idx = np.argmin(kde_vals[pi[0] : pi[1] + 1]) + pi[0]
    return grid[pi], grid[min_idx : min_idx + 1]


h_values = [0.05, h_silver, h_crit, h_crit * 3]
labels = [
    "h = 0.05 (too small)",
    f"h = {h_silver:.4f} (Silverman)",
    f"h = {h_crit:.4f} (critical)",
    f"h = {h_crit * 3:.4f} (too large)",
]

grid = np.linspace(x.min() - 1, x.max() + 1, 1000)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("KDE Bandwidth Sweep", fontsize=14, y=1.02)

for ax, h, label in zip(axes.flat, h_values, labels):
    kde = gaussian_kde(x, grid, h)
    n_modes = count_modes(x, h)

    ax.hist(x, bins=30, density=True, alpha=0.3, color="gray")
    ax.plot(grid, kde, "b-", lw=2)
    px, tx = _peak_trough_marks(x, h, grid, kde)
    if px is not None:
        ax.plot(px, gaussian_kde(x, px, h), "r^", ms=8, label="peaks")
    if tx is not None:
        ax.plot(tx, gaussian_kde(x, tx, h), "go", ms=6, label="trough")
    ax.set_title(f"{label}  |  {n_modes} peak(s)")
    ax.set_xlabel("x")
    ax.set_ylabel("density")
    if px is not None or tx is not None:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print("Figure 1: KDE Bandwidth Sweep - complete")

### 2. Critical Bandwidth Transition

Three overlaid KDE curves showing the bimodal-to-unimodal transition:
- **Blue**: h = h_crit × 0.85 (bimodal — two clear peaks)
- **Green**: h = h_crit (critical — just barely unimodal)
- **Red**: h = h_crit × 1.15 (unimodal — over-smoothed)

The dashed vertical line marks the trough position of the bimodal curve.

In [ ]:
h_bimodal = h_crit * 0.85
h_unimodal = h_crit * 1.15

kde_bi = gaussian_kde(x, grid, h_bimodal)
kde_crit = gaussian_kde(x, grid, h_crit)
kde_uni = gaussian_kde(x, grid, h_unimodal)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(grid, kde_bi, "b-", lw=2, label=f"Bimodal  (h = {h_bimodal:.4f})")
ax.plot(grid, kde_crit, "g-", lw=2, label=f"Critical (h = {h_crit:.4f})")
ax.plot(grid, kde_uni, "r-", lw=2, label=f"Unimodal (h = {h_unimodal:.4f})")

trough = find_trough(x, h_bimodal, refine=True)
if trough is not None:
    y_trough = gaussian_kde(x, np.array([trough]), h_bimodal)[0]
    ax.axvline(trough, color="blue", ls="--", lw=1, alpha=0.6)
    ax.plot(trough, y_trough, "bo", ms=8)

ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Critical Bandwidth Transition", fontsize=13)
ax.legend(fontsize=10)

ax.annotate(
    f"h_crit = {h_crit:.4f}",
    xy=(0, 0),
    xycoords="axes fraction",
    fontsize=10,
    ha="left",
    va="bottom",
    bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="gray", alpha=0.8),
)

plt.tight_layout()
plt.show()
print("Figure 2: Critical Bandwidth Transition - complete")

### 3. Component Decomposition

The bimodal distribution is decomposed into two component Gaussians by
splitting the data at the KDE trough and computing sample statistics for each side.

- **Orange dashed**: Component 1 (lower mean)
- **Green dashed**: Component 2 (higher mean)
- **Red dotted vertical**: Separation point (trough)
- **Red triangles**: KDE peak positions

In [ ]:
def _gaussian_pdf(x, mu, sigma, weight):
    return weight * scipy_stats.norm.pdf(x, mu, sigma)


result = detect_components(x, h_factor=0.85)
h = result.critical_bandwidth * 0.85

kde = gaussian_kde(x, grid, h)

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(x, bins=30, density=True, alpha=0.3, color="gray")
ax.plot(grid, kde, "b-", lw=2.5, label="KDE")

pdf1 = _gaussian_pdf(grid, result.component1.mean, result.component1.std, result.component1.weight)
pdf2 = _gaussian_pdf(grid, result.component2.mean, result.component2.std, result.component2.weight)

ax.plot(
    grid,
    pdf1,
    "orange",
    ls="--",
    lw=2,
    label=f"C1: \u03bc={result.component1.mean:.3f}, \u03c3={result.component1.std:.3f}, w={result.component1.weight:.3f}",
)
ax.plot(
    grid,
    pdf2,
    "green",
    ls="--",
    lw=2,
    label=f"C2: \u03bc={result.component2.mean:.3f}, \u03c3={result.component2.std:.3f}, w={result.component2.weight:.3f}",
)

ax.axvline(
    result.separation_point,
    color="red",
    ls=":",
    lw=2,
    alpha=0.7,
    label=f"Split: x={result.separation_point:.4f}",
)

peaks = signal.find_peaks(kde, prominence=0.01 * np.max(kde))[0]
if len(peaks) >= 2:
    top_two = np.argsort(kde[peaks])[-2:]
    peak_indices = np.sort(peaks[top_two])
    ax.plot(grid[peak_indices], kde[peak_indices], "r^", ms=10)

ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title("Bimodal Component Decomposition", fontsize=13)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print("Figure 3: Component Decomposition - complete")

## Interactive Exploration

To explore further, modify the cells above and re-run them:

| Parameter | Location | Effect |
|-----------|----------|--------|
| seed (42) | Cell 4 (case.generator(42)) | Different random sample |
| benchmark case | Cell 4 (BENCHMARK_CASES[...]) | Different distribution shape |
| h_factor (0.85) | Cell 8 (detect_components(x, h_factor=...)) | Trough depth for decomposition |
| bandwidth values | Cell 6 (h_values) | Which bandwidths to compare |

Available benchmark cases:
- well_separated_equal_var — Two well-separated Gaussians (used above)
- moderate_separation — Closer modes, harder to distinguish
- barely_separated — Nearly touching modes
- unequal_variance — Components with different widths
- unequal_weights — Asymmetric sample sizes
- extreme_separation — Very wide gap between modes
- trimodal — Three modes (outer peaks determine critical bandwidth)

In [ ]:
# Extension exercise: compare dip ratios across benchmark cases
print(f"{'Case':<30} {'Dip Ratio':<12} {'h_crit':<10} {'Converged':<10}")
print("-" * 62)

for name, c in BENCHMARK_CASES.items():
    x_sample = c.generator(42)
    hc, ok = critical_bandwidth(x_sample)
    r = detect_components(x_sample)
    print(f"{name:<30} {r.dip_ratio:<12.4f} {hc:<10.4f} {str(ok):<10}")